# OPV Physics-Informed Gaussian Process Regression

This notebook compares a standard Gaussian Process Regression (GPR) model with three selected physics-informed GPR models for predicting organic photovoltaic (OPV) power conversion efficiency (PCE).

The workflow uses repeated low-data 20 percent training / 80 percent test splits, learning curves at 10 and 20 percent of the full dataset, uncertainty-aware parity plots, a production model trained on 100 percent of the data, and SHAP analysis for the selected best model.

## Paper Reference

This example uses the dataset associated with:

H. Sahu, W. Rao, A. Troisi, and H. Ma, **"Toward Predicting Efficiency of Organic Solar Cells via Machine Learning and Improved Descriptors,"** *Advanced Energy Materials*, 8(24), 1801032, 2018. DOI: [10.1002/aenm.201801032](https://doi.org/10.1002/aenm.201801032).

The key physical message used here is that PCE depends on multiple microscopic processes, and the paper highlights the importance of orbitals energetically close to HOMO and LUMO. In particular, small donor `Delta_H`, donor `Delta_L`, and acceptor `Delta_LA` can allow orbitals beyond HOMO/LUMO to participate in exciton formation, exciton dissociation, and charge transport.

## Physics-Informed Mean Functions Tested

A low-data screening run evaluated several candidate physics-informed mean functions with 20 random stratified 20 percent training / 80 percent test splits. The retained models are the three physics-informed candidates with the lowest RMSE while also considering RMSE standard deviation across splits.

| Model | Mean-function information | Physical motivation |
| --- | --- | --- |
| Standard GPR | learned constant mean | data-only baseline |
| PI-GPR: degeneracy + binding | frontier-orbital near-degeneracy and low exciton binding | compact physics prior focused on orbital participation and charge separation |
| PI-GPR: degeneracy + binding + transport | frontier-orbital near-degeneracy, low exciton binding, and transport proxies | adds charge-transport information to the compact physics prior |
| PI-GPR: degeneracy | frontier-orbital near-degeneracy | tests the source paper's central orbital-participation hypothesis |

The final notebook keeps only these four models so the comparison stays focused on the low-data regime.

## 1. Setup

The package is still local to this repository. This setup cell finds the local package whether the notebook is run from the workspace-level `examples/opv` folder or from an `examples/opv` folder inside the package repository.

In [ ]:
from pathlib import Path
import os
import sys

# Helpful on local macOS scientific Python stacks that load multiple OpenMP runtimes.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

NOTEBOOK_DIR = Path.cwd().resolve()


def find_project_root(start: Path) -> Path:
    """Find the local package root containing pyproject.toml and matgpr."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "matgpr").exists():
            return candidate
        sibling = candidate / "matgpr"
        if (sibling / "pyproject.toml").exists() and (sibling / "matgpr").exists():
            return sibling
    raise FileNotFoundError("Could not find the local matgpr package root.")


PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Using package from: {PROJECT_ROOT}")

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import torch
from gpytorch.utils.warnings import GPInputWarning
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from matgpr import (
    PhysicsInformedMean,
    drop_duplicate_rows,
    fit_gpytorch_gpr,
    normalize_column_names,
    plot_correlation_matrix,
    plot_distribution,
    plot_parity,
    regression_metrics,
    replace_missing_placeholders,
    summarize_missingness,
    summarize_numeric_columns,
)

RANDOM_STATE = 42
plt.style.use("default")
plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 12,
        "legend.fontsize": 9,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)
torch.set_num_threads(1)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=GPInputWarning)

## 2. Load and Clean the Dataset

The original file uses names such as `AL-DH` and `DL-AL`. The workflow keeps a descriptor table with the original names, then normalizes dataframe columns so the modeling code can use stable Python-friendly names.

In [ ]:
def find_data_path(start: Path) -> Path:
    """Find dataset.csv in the OPV example folder inside this repository."""
    candidates = [start / "dataset.csv", PROJECT_ROOT / "examples" / "opv" / "dataset.csv"]
    candidates += [parent / "examples" / "opv" / "dataset.csv" for parent in start.parents]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find examples/opv/dataset.csv")


DATA_PATH = find_data_path(NOTEBOOK_DIR)
raw_df = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH}")
raw_df.head()

In [ ]:
descriptor_info = pd.DataFrame(
    [
        ("polarizability", "polarizability", "Polarizability of donor molecules"),
        ("delLA", "della", "LUMO/LUMO+1 energetic difference of acceptors"),
        ("delLD", "delld", "LUMO/LUMO+1 energetic difference of donors"),
        ("N_atom", "n_atom", "Number of unsaturated atoms in the donor main conjugation path"),
        ("Eg", "eg", "Strongest singlet excited-state transition energy"),
        ("lamda_h", "lamda_h", "Hole reorganization energy in donor molecules"),
        ("DIP", "dip", "Vertical ionization potential of donor molecules"),
        ("AL-DH", "al_dh", "Donor HOMO / acceptor LUMO energetic difference"),
        ("delHD", "delhd", "HOMO/HOMO-1 energetic difference of donors"),
        ("E_bind", "e_bind", "Hole-electron binding energy in donor molecules"),
        ("DL-AL", "dl_al", "Donor LUMO / acceptor LUMO energetic difference"),
        ("delGE", "delge", "Dipole-moment change from ground state to first excited state"),
        ("E_T1", "e_t1", "Lowest triplet excited-state transition energy"),
    ],
    columns=["paper_column", "model_column", "description"],
)

descriptor_info

In [ ]:
df = (
    raw_df
    .pipe(replace_missing_placeholders)
    .pipe(normalize_column_names)
    .pipe(drop_duplicate_rows)
)

feature_columns = descriptor_info["model_column"].tolist()
target_column = "pce"

missing_columns = [column for column in feature_columns + [target_column] if column not in df.columns]
if missing_columns:
    raise KeyError(f"Missing expected columns after cleaning: {missing_columns}")

model_df = df[[target_column] + feature_columns].copy()
print(f"Rows: {model_df.shape[0]}")
print(f"Features: {len(feature_columns)}")
summarize_missingness(model_df)

In [ ]:
summarize_numeric_columns(model_df).round(3)

## 3. Quick Data Checks

These plots are not used for fitting, but they help confirm that the target and descriptor space look reasonable before building physics-informed models.

In [ ]:
fig, ax = plot_distribution(
    model_df[target_column],
    bins=20,
    title="Distribution of OPV Power Conversion Efficiency",
    xlabel="PCE (%)",
)
plt.show()

In [ ]:
fig, ax, corr = plot_correlation_matrix(
    model_df,
    columns=[target_column] + feature_columns,
    figsize=(10, 8),
    title="Correlation Matrix for OPV Descriptors and PCE",
    annotate=False,
)
plt.show()

In [ ]:
correlation_with_pce = (
    model_df.corr(numeric_only=True)[target_column]
    .drop(target_column)
    .sort_values(key=lambda values: values.abs(), ascending=False)
)
correlation_with_pce.to_frame("linear_correlation_with_pce").round(3)

## 4. Physics-Informed Mean Functions

Each equation returns a mean PCE in the original target units. The GP then learns residual deviations around that mean. The equations use standardized physical scores so the learnable weights have a simple interpretation: positive weights reward physically favorable descriptors.

In [ ]:
def zscore_feature(features, parameters, name):
    """Return a standardized feature tensor using training-set statistics."""
    return (features[name] - parameters[f"{name}_mean"]) / parameters[f"{name}_std"]


def frontier_degeneracy_score(features, parameters):
    """Higher score means smaller frontier-orbital gaps."""
    donor_h = -zscore_feature(features, parameters, "delhd")
    donor_l = -zscore_feature(features, parameters, "delld")
    acceptor_l = -zscore_feature(features, parameters, "della")
    return (donor_h + donor_l + acceptor_l) / 3.0


def exciton_separation_score(features, parameters):
    """Higher score means lower hole-electron binding energy."""
    return -zscore_feature(features, parameters, "e_bind")


def transport_score(features, parameters):
    """Higher score means larger conjugation/polarizability and lower reorganization energy."""
    log_n_atom = torch.log(torch.clamp(features["n_atom"], min=1.0))
    log_polarizability = torch.log(torch.clamp(features["polarizability"], min=1.0))
    conjugation = (log_n_atom - parameters["log_n_atom_mean"]) / parameters["log_n_atom_std"]
    polarizability = (
        (log_polarizability - parameters["log_polarizability_mean"])
        / parameters["log_polarizability_std"]
    )
    reorganization = -zscore_feature(features, parameters, "lamda_h")
    return (conjugation + polarizability + reorganization) / 3.0


def optical_gap_window_score(features, parameters):
    """Higher score means the optical gap is close to the training-set center."""
    return -torch.abs(zscore_feature(features, parameters, "eg"))


def energy_alignment_score(features, parameters):
    """Higher score rewards donor-acceptor offsets used as charge-transfer proxies."""
    voltage_proxy = zscore_feature(features, parameters, "al_dh")
    lumo_offset = zscore_feature(features, parameters, "dl_al")
    return (voltage_proxy + lumo_offset) / 2.0


def excited_state_score(features, parameters):
    """Higher score combines excited-state dipole change and triplet-energy proxies."""
    dipole_response = zscore_feature(features, parameters, "delge")
    triplet_energy = zscore_feature(features, parameters, "e_t1")
    return (dipole_response + triplet_energy) / 2.0


PHYSICS_SCORE_FUNCTIONS = {
    "degeneracy": frontier_degeneracy_score,
    "binding": exciton_separation_score,
    "transport": transport_score,
    "optical_gap": optical_gap_window_score,
    "energy_alignment": energy_alignment_score,
    "excited_state": excited_state_score,
}

PHYSICS_SCORE_FEATURES = {
    "degeneracy": ["delhd", "delld", "della"],
    "binding": ["e_bind"],
    "transport": ["n_atom", "polarizability", "lamda_h"],
    "optical_gap": ["eg"],
    "energy_alignment": ["al_dh", "dl_al"],
    "excited_state": ["delge", "e_t1"],
}


def unique_features_for_scores(score_names):
    """Return unique feature names required by a list of physics scores."""
    features = []
    for score_name in score_names:
        for feature in PHYSICS_SCORE_FEATURES[score_name]:
            if feature not in features:
                features.append(feature)
    return features


def make_additive_physics_equation(score_names):
    """Build an additive physics mean from named score functions."""
    score_names = tuple(score_names)

    def equation(features, parameters):
        mean = parameters["baseline"]
        for score_name in score_names:
            weight = parameters[f"{score_name}_weight"]
            mean = mean + weight * PHYSICS_SCORE_FUNCTIONS[score_name](features, parameters)
        return mean

    equation.__name__ = "physics_mean_" + "_".join(score_names)
    return equation

In [ ]:
def physics_model_spec(label, score_names, description):
    """Create a notebook model specification for an additive physics mean."""
    return {
        "label": label,
        "equation": make_additive_physics_equation(score_names),
        "score_names": list(score_names),
        "features": unique_features_for_scores(score_names),
        "weights": [f"{score_name}_weight" for score_name in score_names],
        "description": description,
    }


SCREENING_RESULTS_20_PERCENT = pd.DataFrame(
    [
        {
            "model_key": "pi_gpr_degeneracy_binding",
            "model": "PI-GPR: degeneracy + binding",
            "test_RMSE_mean": 1.3239,
            "test_RMSE_std": 0.0646,
            "test_R2_mean": 0.3780,
        },
        {
            "model_key": "pi_gpr_degeneracy_binding_transport",
            "model": "PI-GPR: degeneracy + binding + transport",
            "test_RMSE_mean": 1.3298,
            "test_RMSE_std": 0.0758,
            "test_R2_mean": 0.3720,
        },
        {
            "model_key": "pi_gpr_degeneracy",
            "model": "PI-GPR: degeneracy",
            "test_RMSE_mean": 1.3361,
            "test_RMSE_std": 0.0634,
            "test_R2_mean": 0.3669,
        },
        {
            "model_key": "pi_gpr_all_simple",
            "model": "PI-GPR: all simple physics",
            "test_RMSE_mean": 1.3603,
            "test_RMSE_std": 0.0910,
            "test_R2_mean": 0.3414,
        },
        {
            "model_key": "pi_gpr_transport",
            "model": "PI-GPR: transport",
            "test_RMSE_mean": 1.4107,
            "test_RMSE_std": 0.0533,
            "test_R2_mean": 0.2944,
        },
        {
            "model_key": "pi_gpr_binding",
            "model": "PI-GPR: binding",
            "test_RMSE_mean": 1.4105,
            "test_RMSE_std": 0.0569,
            "test_R2_mean": 0.2944,
        },
        {
            "model_key": "standard_gpr",
            "model": "Standard GPR",
            "test_RMSE_mean": 1.5229,
            "test_RMSE_std": 0.0732,
            "test_R2_mean": 0.1776,
        },
        {
            "model_key": "pi_gpr_energy_alignment",
            "model": "PI-GPR: energy alignment",
            "test_RMSE_mean": 1.5283,
            "test_RMSE_std": 0.0860,
            "test_R2_mean": 0.1712,
        },
        {
            "model_key": "pi_gpr_optical_gap",
            "model": "PI-GPR: optical gap",
            "test_RMSE_mean": 1.5282,
            "test_RMSE_std": 0.0922,
            "test_R2_mean": 0.1713,
        },
    ]
)
SCREENING_RESULTS_20_PERCENT["selection_score"] = (
    SCREENING_RESULTS_20_PERCENT["test_RMSE_mean"]
    + 0.25 * SCREENING_RESULTS_20_PERCENT["test_RMSE_std"]
)
SCREENING_RESULTS_20_PERCENT = SCREENING_RESULTS_20_PERCENT.sort_values(
    ["selection_score", "test_RMSE_mean", "test_RMSE_std"]
).reset_index(drop=True)

PHYSICS_MODEL_SPECS = {
    "standard_gpr": {
        "label": "Standard GPR",
        "equation": None,
        "score_names": [],
        "features": [],
        "weights": [],
        "description": "Constant mean baseline with a learned GP covariance.",
    },
    "pi_gpr_degeneracy_binding": physics_model_spec(
        "PI-GPR: degeneracy + binding",
        ["degeneracy", "binding"],
        "Compact prior combining orbital near-degeneracy with low exciton binding energy.",
    ),
    "pi_gpr_degeneracy_binding_transport": physics_model_spec(
        "PI-GPR: degeneracy + binding + transport",
        ["degeneracy", "binding", "transport"],
        "Combines near-degeneracy, low exciton binding energy, and transport proxies.",
    ),
    "pi_gpr_degeneracy": physics_model_spec(
        "PI-GPR: degeneracy",
        ["degeneracy"],
        "Rewards small donor/acceptor frontier-orbital gaps.",
    ),
}
SELECTED_MODEL_KEYS = list(PHYSICS_MODEL_SPECS)

SCREENING_RESULTS_20_PERCENT.round(4)

In [ ]:
def physics_statistics(X_train):
    """Training-set statistics used inside the physics equations."""
    stats = {}
    for column in feature_columns:
        values = X_train[column].astype(float)
        std = float(values.std(ddof=0))
        stats[f"{column}_mean"] = float(values.mean())
        stats[f"{column}_std"] = std if std > 0 else 1.0

    for column in ["n_atom", "polarizability"]:
        log_values = np.log(np.clip(X_train[column].to_numpy(dtype=float), 1.0, None))
        std = float(log_values.std(ddof=0))
        stats[f"log_{column}_mean"] = float(log_values.mean())
        stats[f"log_{column}_std"] = std if std > 0 else 1.0

    return stats


def build_physics_mean(model_key, X_train, y_train, feature_scaler):
    """Create the requested PhysicsInformedMean for scaled GPyTorch inputs."""
    spec = PHYSICS_MODEL_SPECS[model_key]
    if spec["equation"] is None:
        return None

    learnable_parameters = {"baseline": float(np.mean(y_train))}
    learnable_parameters.update({weight: 0.25 for weight in spec["weights"]})

    feature_indices = {column: feature_columns.index(column) for column in spec["features"]}
    feature_means = {
        column: float(feature_scaler.mean_[feature_columns.index(column)])
        for column in spec["features"]
    }
    feature_stds = {
        column: float(feature_scaler.scale_[feature_columns.index(column)])
        for column in spec["features"]
    }

    return PhysicsInformedMean(
        equation=spec["equation"],
        feature_indices=feature_indices,
        learnable_parameters=learnable_parameters,
        positive_parameters=tuple(spec["weights"]),
        fixed_parameters=physics_statistics(X_train),
        feature_means=feature_means,
        feature_stds=feature_stds,
    )

## 5. Low-Data Train/Test Splits

The example is intentionally set up as a low-data problem. Each repeated split uses 20 percent of the full dataset for training and 80 percent as an external test set. Learning-curve x-axis values are reported as the percentage of the full dataset used for training: 10 percent and 20 percent.

In [ ]:
X = model_df[feature_columns].copy()
y = model_df[target_column].copy()

TEST_SIZE = 0.80
MAX_TRAIN_PERCENT = int(round((1.0 - TEST_SIZE) * 100))
TEST_PERCENT = int(round(TEST_SIZE * 100))

split_bins = pd.qcut(y, q=5, labels=False, duplicates="drop")
X_train_pool, X_test, y_train_pool, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=split_bins,
)

print(f"Total rows: {len(X)}")
print(f"Training pool rows ({MAX_TRAIN_PERCENT}%): {len(X_train_pool)}")
print(f"External test rows ({TEST_PERCENT}%): {len(X_test)}")

In [ ]:
def stratified_fraction_indices(y_values, fraction, random_state, n_bins=5):
    """Sample a fraction of rows while preserving the target distribution approximately."""
    if fraction >= 1.0:
        return y_values.index

    bins = pd.qcut(y_values, q=n_bins, labels=False, duplicates="drop")
    sample_frame = pd.DataFrame({"bin": bins}, index=y_values.index)
    sampled = sample_frame.groupby("bin", group_keys=False).sample(
        frac=fraction,
        random_state=random_state,
    )
    return sampled.index


def root_mean_squared_error(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

## 6. Learning Curve Experiment

The learning curve uses 10 percent and 20 percent of the full dataset as training data. Each point is repeated over 20 stratified random 20/80 train/test splits. The 10 percent point samples half of each 20 percent training split, while the 20 percent point uses the full training split.

This setup is designed to highlight whether physics-informed priors improve GPR performance in the low-data regime.

In [ ]:
TRAIN_PERCENTS = list(range(10, MAX_TRAIN_PERCENT + 1, 10))
TRAIN_FRACTIONS_BY_PERCENT = {
    train_percent: train_percent / MAX_TRAIN_PERCENT
    for train_percent in TRAIN_PERCENTS
}
SELECTION_TRAIN_PERCENT = 20
N_REPEATS = 20
TRAINING_ITER = 60
PRODUCTION_TRAINING_ITER = 120
LEARNING_RATE = 0.05
KERNEL = "matern"
MODEL_ORDER = SELECTED_MODEL_KEYS

n_total_fits = len(TRAIN_PERCENTS) * N_REPEATS * len(MODEL_ORDER)
print(f"Models: {[PHYSICS_MODEL_SPECS[key]['label'] for key in MODEL_ORDER]}")
print(f"Training data percentages: {TRAIN_PERCENTS}")
print(f"Random 20/80 splits per percentage: {N_REPEATS}")
print(f"GPR optimization iterations per fit: {TRAINING_ITER}")
print(f"Total GPR fits: {n_total_fits}")

In [ ]:
def fit_gp_model(model_key, X_train_subset, y_train_subset, *, training_iter=TRAINING_ITER):
    """Fit one GPyTorch GPR model and keep its feature scaler."""
    scaler = StandardScaler().fit(X_train_subset)
    X_train_scaled = scaler.transform(X_train_subset).copy()

    mean_module = build_physics_mean(
        model_key,
        X_train_subset,
        y_train_subset,
        scaler,
    )

    result = fit_gpytorch_gpr(
        X_train_scaled,
        y_train_subset.to_numpy(),
        kernel=KERNEL,
        ard=True,
        mean_module=mean_module,
        lr=LEARNING_RATE,
        training_iter=training_iter,
        initial_noise=0.1,
        standardize_y=True,
        verbose=False,
    )

    return {
        "model_key": model_key,
        "label": PHYSICS_MODEL_SPECS[model_key]["label"],
        "result": result,
        "scaler": scaler,
        "train_size": len(X_train_subset),
    }


def predict_with_fitted_gp(fitted_model, X_eval, *, return_std=True):
    """Predict with a fitted model dictionary returned by fit_gp_model."""
    X_eval_scaled = fitted_model["scaler"].transform(X_eval[feature_columns]).copy()
    return fitted_model["result"].predict(X_eval_scaled, return_std=return_std)


def fit_and_score_gp(model_key, X_train_subset, y_train_subset, X_eval, y_eval):
    """Fit one GPyTorch GPR model and score it on the external test set."""
    fitted_model = fit_gp_model(model_key, X_train_subset, y_train_subset)
    prediction = predict_with_fitted_gp(fitted_model, X_eval, return_std=True)
    metrics = regression_metrics(y_eval, prediction.mean, prefix="test")
    metrics = {
        "test_RMSE": metrics["test_RMSE"],
        "test_R2": metrics["test_R2"],
        "test_MAE": metrics["test_MAE"],
        "test_r": metrics["test_r"],
    }
    return fitted_model, prediction, metrics

In [ ]:
learning_curve_records = []
final_fit_cache = {}
completed_fits = 0

for train_percent in TRAIN_PERCENTS:
    train_fraction = TRAIN_FRACTIONS_BY_PERCENT[train_percent]
    for repeat in range(N_REPEATS):
        split_seed = RANDOM_STATE + repeat
        X_train_pool_repeat, X_test_repeat, y_train_pool_repeat, y_test_repeat = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            random_state=split_seed,
            stratify=split_bins,
        )
        subset_seed = RANDOM_STATE + repeat + train_percent * 100
        subset_index = stratified_fraction_indices(
            y_train_pool_repeat,
            fraction=train_fraction,
            random_state=subset_seed,
        )
        X_subset = X_train_pool_repeat.loc[subset_index]
        y_subset = y_train_pool_repeat.loc[subset_index]

        for model_key in MODEL_ORDER:
            fitted_model, prediction, metrics = fit_and_score_gp(
                model_key,
                X_subset,
                y_subset,
                X_test_repeat,
                y_test_repeat,
            )
            completed_fits += 1
            record = {
                "model_key": model_key,
                "model": PHYSICS_MODEL_SPECS[model_key]["label"],
                "train_fraction_of_pool": train_fraction,
                "train_percent": train_percent,
                "train_size": len(X_subset),
                "test_size": len(X_test_repeat),
                "repeat": repeat,
                **metrics,
            }
            learning_curve_records.append(record)

            if train_percent == MAX_TRAIN_PERCENT and repeat == 0:
                final_fit_cache[model_key] = {
                    **fitted_model,
                    "prediction": prediction,
                }

            if completed_fits % 50 == 0 or completed_fits == n_total_fits:
                print(f"Completed {completed_fits}/{n_total_fits} fits")

learning_curve = pd.DataFrame(learning_curve_records)
learning_curve.head()

In [ ]:
learning_summary = (
    learning_curve
    .groupby(["model_key", "model", "train_percent"], as_index=False)
    .agg(
        train_size_mean=("train_size", "mean"),
        train_size_std=("train_size", "std"),
        test_RMSE_mean=("test_RMSE", "mean"),
        test_RMSE_std=("test_RMSE", "std"),
        test_R2_mean=("test_R2", "mean"),
        test_R2_std=("test_R2", "std"),
        test_r_mean=("test_r", "mean"),
        test_r_std=("test_r", "std"),
    )
)

available_train_percents = sorted(learning_summary["train_percent"].unique())
selection_train_percent_used = min(
    available_train_percents,
    key=lambda value: abs(value - SELECTION_TRAIN_PERCENT),
)
if selection_train_percent_used != SELECTION_TRAIN_PERCENT:
    print(
        f"Selection train percent {SELECTION_TRAIN_PERCENT}% is unavailable; "
        f"using {selection_train_percent_used}% instead."
    )

model_selection_table = (
    learning_summary[learning_summary["train_percent"] == selection_train_percent_used]
    .copy()
    .sort_values(["test_RMSE_mean", "test_RMSE_std"], ascending=[True, True])
)
model_selection_table["selection_score"] = (
    model_selection_table["test_RMSE_mean"]
    + 0.25 * model_selection_table["test_RMSE_std"].fillna(0.0)
)
model_selection_table = model_selection_table.sort_values(
    ["selection_score", "test_RMSE_mean", "test_RMSE_std"]
).reset_index(drop=True)

best_model_key = model_selection_table.iloc[0]["model_key"]
best_physics_model_key = model_selection_table.query(
    "model_key != 'standard_gpr'"
).iloc[0]["model_key"]

print(
    f"Best model at {selection_train_percent_used}% training data:",
    PHYSICS_MODEL_SPECS[best_model_key]["label"],
)
print(
    f"Best PI model at {selection_train_percent_used}% training data:",
    PHYSICS_MODEL_SPECS[best_physics_model_key]["label"],
)

learning_summary[[
    "model",
    "train_percent",
    "train_size_mean",
    "test_RMSE_mean",
    "test_RMSE_std",
    "test_R2_mean",
    "test_R2_std",
]].round(3)

## 7. Learning Curve Plots

The first plot shows external-test RMSE, where lower is better. The second plot shows external-test R2, where higher is better. Each point is the mean across 20 stratified random subsets and the error bar is one standard deviation.

In [ ]:
MODEL_STYLES = {
    "standard_gpr": {"color": "#4D4D4D", "marker": "o"},
    "pi_gpr_degeneracy_binding": {"color": "#009E73", "marker": "D"},
    "pi_gpr_degeneracy_binding_transport": {"color": "#0072B2", "marker": "s"},
    "pi_gpr_degeneracy": {"color": "#D55E00", "marker": "^"},
}


def plot_learning_curve_metric(mean_column, std_column, ylabel, title, *, lower_is_better=False):
    fig, ax = plt.subplots(figsize=(7.2, 5.0))

    for model_key in MODEL_ORDER:
        model_data = learning_summary[
            learning_summary["model_key"] == model_key
        ].sort_values("train_percent")
        label = PHYSICS_MODEL_SPECS[model_key]["label"]
        style = MODEL_STYLES[model_key]

        ax.errorbar(
            model_data["train_percent"],
            model_data[mean_column],
            yerr=model_data[std_column].fillna(0.0),
            marker=style["marker"],
            linewidth=2.0,
            markersize=5.5,
            capsize=3,
            elinewidth=1.1,
            color=style["color"],
            label=label,
        )

    ax.set_title(title, pad=8)
    ax.set_xlabel("Training data (% of full dataset)")
    ax.set_ylabel(ylabel)
    ax.set_xticks(TRAIN_PERCENTS)
    ax.grid(True, alpha=0.25, linewidth=0.7)
    if not lower_is_better:
        ax.axhline(0.0, color="black", linewidth=1.0, linestyle="--", alpha=0.35)
        ax.set_ylim(top=1.05)
    ax.legend(frameon=False, loc="best")
    fig.tight_layout()
    return fig, ax


rmse_fig, rmse_ax = plot_learning_curve_metric(
    "test_RMSE_mean",
    "test_RMSE_std",
    "RMSE in PCE (%)",
    "External-Test RMSE Learning Curve",
    lower_is_better=True,
)

r2_fig, r2_ax = plot_learning_curve_metric(
    "test_R2_mean",
    "test_R2_std",
    r"$R^2$ on external-test PCE",
    r"External-Test $R^2$ Learning Curve",
)

plt.show()

## 8. External-Test Parity and Uncertainty

The following figure compares all four retained models trained on the fixed 20 percent training split and evaluated on the fixed 80 percent external test set. Error bars show the GP predictive standard deviation for each test prediction.

In [ ]:
parity_predictions = {
    model_key: final_fit_cache[model_key]["prediction"]
    for model_key in MODEL_ORDER
}
all_predicted_values = np.concatenate(
    [prediction.mean for prediction in parity_predictions.values()]
)
min_value = min(y_test.min(), all_predicted_values.min()) - 0.5
max_value = max(y_test.max(), all_predicted_values.max()) + 0.5

fig, axes = plt.subplots(2, 2, figsize=(10.5, 9.2), sharex=True, sharey=True)
axes = axes.ravel()

for ax, model_key in zip(axes, MODEL_ORDER):
    prediction = parity_predictions[model_key]
    metrics = regression_metrics(y_test, prediction.mean)
    style = MODEL_STYLES[model_key]

    ax.errorbar(
        y_test,
        prediction.mean,
        yerr=prediction.std,
        fmt=style["marker"],
        markersize=5.2,
        alpha=0.82,
        capsize=2,
        elinewidth=0.8,
        color=style["color"],
        markeredgecolor="black",
        markeredgewidth=0.35,
    )
    ax.plot([min_value, max_value], [min_value, max_value], "--", color="black", linewidth=1.1)
    ax.set_xlim(min_value, max_value)
    ax.set_ylim(min_value, max_value)
    ax.set_title(PHYSICS_MODEL_SPECS[model_key]["label"], pad=7)
    ax.grid(True, alpha=0.22, linewidth=0.7)
    ax.text(
        0.05,
        0.95,
        f"RMSE = {root_mean_squared_error(y_test, prediction.mean):.2f}\n"
        f"R2 = {metrics['R2']:.2f}\n"
        f"r = {metrics['r']:.2f}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=9.5,
        bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "0.75", "alpha": 0.92},
    )

for ax in axes[2:]:
    ax.set_xlabel("Experimental PCE (%)")
for ax in axes[::2]:
    ax.set_ylabel("Predicted PCE (%)")

fig.suptitle("External-Test Parity with GP Predictive Uncertainty", y=0.995)
fig.tight_layout()
plt.show()

## 9. Learned Physics Mean Parameters

The learned positive weights show how strongly each retained physics prior term is used after joint optimization with the GP covariance. They are not causal coefficients, but they are useful diagnostics for whether the prior mean is active.

In [ ]:
parameter_rows = []
for model_key, cached in final_fit_cache.items():
    mean_module = cached["result"].model.mean_module
    if hasattr(mean_module, "current_parameter_values"):
        values = mean_module.current_parameter_values()
        parameter_rows.extend(
            {
                "model": PHYSICS_MODEL_SPECS[model_key]["label"],
                "parameter": name,
                "value": value,
            }
            for name, value in values.items()
        )

physics_parameter_table = pd.DataFrame(parameter_rows).round(4)
physics_parameter_table

## 10. Production Model on 100 Percent of the Data

The production model is selected from the retained four-model comparison using the 20 percent training-data RMSE summary. It is then refit using all 280 OPV records. This model is intended for interpretation and future prediction workflows where no external test set is being held back.

In [ ]:
production_model_key = best_model_key
production_model_label = PHYSICS_MODEL_SPECS[production_model_key]["label"]

if production_model_key == "standard_gpr":
    print(
        "Standard GPR was selected by the 20% RMSE criterion. "
        "The best retained physics-informed model is "
        f"{PHYSICS_MODEL_SPECS[best_physics_model_key]['label']}."
    )
else:
    print(f"Selected production model: {production_model_label}")

production_fit = fit_gp_model(
    production_model_key,
    X,
    y,
    training_iter=PRODUCTION_TRAINING_ITER,
)
production_training_prediction = predict_with_fitted_gp(production_fit, X, return_std=True)
production_training_metrics = regression_metrics(y, production_training_prediction.mean)

pd.DataFrame(
    [
        {
            "production_model": production_model_label,
            "training_rows": len(X),
            "RMSE": root_mean_squared_error(y, production_training_prediction.mean),
            "R2": production_training_metrics["R2"],
            "r": production_training_metrics["r"],
        }
    ]
).round(3)

## 11. SHAP Analysis for the Production Model

SHAP values are computed with a model-agnostic permutation explainer using the production model prediction function. The bar plot ranks features by mean absolute SHAP value, and the dependence plots show how the top features move predictions up or down.

In [ ]:
def production_predict(input_data):
    """Prediction function used by SHAP. Returns PCE predictions in original units."""
    if isinstance(input_data, pd.DataFrame):
        input_frame = input_data[feature_columns].copy()
    else:
        input_frame = pd.DataFrame(input_data, columns=feature_columns)
    return predict_with_fitted_gp(production_fit, input_frame, return_std=False).mean


SHAP_BACKGROUND_SIZE = min(60, len(X))
SHAP_EXPLAIN_SIZE = min(120, len(X))
shap_background = shap.sample(X[feature_columns], SHAP_BACKGROUND_SIZE, random_state=RANDOM_STATE)
shap_explain = shap.sample(X[feature_columns], SHAP_EXPLAIN_SIZE, random_state=RANDOM_STATE + 1)

explainer = shap.PermutationExplainer(
    production_predict,
    shap_background,
    feature_names=feature_columns,
    seed=RANDOM_STATE,
)
shap_values = explainer(
    shap_explain,
    max_evals=2 * len(feature_columns) + 1,
    batch_size=32,
)

shap_value_array = np.asarray(shap_values.values)
shap_importance = (
    pd.DataFrame(
        {
            "feature": feature_columns,
            "mean_abs_shap": np.abs(shap_value_array).mean(axis=0),
        }
    )
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
shap_importance.head(10).round(4)

In [ ]:
feature_descriptions = descriptor_info.set_index("model_column")["description"].to_dict()

def feature_label(feature):
    """Compact publication label for feature plots."""
    paper_name = descriptor_info.set_index("model_column").loc[feature, "paper_column"]
    return f"{paper_name} ({feature})"


top_features = shap_importance.head(6)["feature"].tolist()
bar_data = shap_importance.head(10).iloc[::-1]

fig, ax = plt.subplots(figsize=(7.0, 5.2))
ax.barh(
    [feature_label(feature) for feature in bar_data["feature"]],
    bar_data["mean_abs_shap"],
    color="#0072B2",
    alpha=0.9,
)
ax.set_xlabel("Mean absolute SHAP value (PCE %)")
ax.set_title(f"Top Features for {production_model_label}")
ax.grid(True, axis="x", alpha=0.25, linewidth=0.7)
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(12.0, 7.2))
axes = axes.ravel()
for ax, feature in zip(axes, top_features):
    feature_index = feature_columns.index(feature)
    scatter = ax.scatter(
        shap_explain[feature],
        shap_value_array[:, feature_index],
        c=production_predict(shap_explain),
        cmap="viridis",
        s=34,
        alpha=0.82,
        edgecolor="black",
        linewidth=0.25,
    )
    ax.axhline(0.0, color="black", linewidth=1.0, linestyle="--", alpha=0.45)
    ax.set_xlabel(feature_label(feature))
    ax.set_ylabel("SHAP value (PCE %)")
    ax.set_title(feature_descriptions.get(feature, feature), fontsize=10)
    ax.grid(True, alpha=0.22, linewidth=0.7)

for ax in axes[len(top_features):]:
    ax.axis("off")

cbar = fig.colorbar(scatter, ax=axes.tolist(), shrink=0.88, pad=0.02)
cbar.set_label("Predicted PCE (%)")
fig.suptitle("Feature Impact on Production-Model Predictions", y=0.99)
fig.subplots_adjust(left=0.08, right=0.88, bottom=0.09, top=0.88, wspace=0.35, hspace=0.55)
plt.show()
plt.close(fig)

plt.figure(figsize=(7.2, 5.6))
shap.summary_plot(
    shap_values,
    shap_explain,
    feature_names=[feature_label(feature) for feature in feature_columns],
    show=False,
    max_display=10,
)
summary_fig = plt.gcf()
summary_fig.set_size_inches(7.2, 5.6)
plt.show()

## 12. Interpretation

The retained comparison is intentionally compact: Standard GPR plus the three physics-informed candidates that performed best under repeated 20/80 low-data splits. The learning curves show whether those physics priors improve low-data RMSE and R2. The four-panel parity figure then compares calibration and predictive uncertainty on a fixed 80 percent test set.

For deployment, the selected best model is refit on all available OPV data. The SHAP analysis should be used to check whether the production model relies on chemically meaningful descriptors and whether the learned feature impacts agree with OPV intuition.